# CML 4 - Linear Regression Assumptions

**Scaler CML | Applied ML (Intro)**

Diagnose the California housing OLS from CML 3. Predictions can look fine while **inference** (coefficients, CIs, p-values) breaks. Assumptions live on the errors; we check them with **residuals**.

**Lab flow**

1. Load data, split first, fit an sklearn `Pipeline` (impute + scale + `LinearRegression`)
2. Residuals vs fitted (linearity + equal variance)
3. Residuals vs a key feature (`MedInc`)
4. Histogram + Q-Q (normality)
5. Correlation heatmap + VIF sketch (multicollinearity)
6. Optional: log-target refit to calm a fan-shaped residual plot


## 0. Setup, load, split, quick OLS

Reuse CML 2/3 hygiene: **split before** fitting the imputer/scaler. Numeric features only for this diagnostic lab.

Data source preference:

1. Local `data/california_housing.csv` if present (CML 3 layout)
2. Else `sklearn.datasets.fetch_california_housing`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.dpi"] = 110


In [ ]:
def load_housing_frame():
    """Prefer local CSV used in CML 3; fall back to sklearn fetch."""
    candidates = [
        Path("data/california_housing.csv"),
        Path("../data/california_housing.csv"),
        Path("/workspace/cml_sessions/s4/data/california_housing.csv"),
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            rename = {
                "median_house_value": "MedHouseVal",
                "median_income": "MedInc",
                "housing_median_age": "HouseAge",
                "total_rooms": "AveRooms_raw",
                "total_bedrooms": "AveBedrms_raw",
                "population": "Population",
                "households": "AveOccup_raw",
                "latitude": "Latitude",
                "longitude": "Longitude",
            }
            df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
            print(f"Loaded local CSV: {path}")
            return df, "local_csv"

    from sklearn.datasets import fetch_california_housing

    bunch = fetch_california_housing(as_frame=True)
    df = bunch.frame.copy()
    print("Loaded sklearn fetch_california_housing")
    return df, "sklearn"


df, source = load_housing_frame()
df.head()


In [ ]:
# Target column: sklearn uses MedHouseVal; some CSVs use median_house_value (renamed above)
TARGET_CANDIDATES = ["MedHouseVal", "median_house_value"]
TARGET = next(c for c in TARGET_CANDIDATES if c in df.columns)

y = df[TARGET]
X = df.drop(columns=[TARGET])
num_cols = X.select_dtypes(include=np.number).columns.tolist()
X = X[num_cols].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

prep = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
model = Pipeline([("prep", prep), ("lr", LinearRegression())])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
resid = y_test.to_numpy() - y_pred

print(f"source={source}  target={TARGET}  n_train={len(X_train)}  n_test={len(X_test)}")
print(
    "holdout  MAE={:.3f}  RMSE={:.3f}  R2={:.3f}".format(
        mean_absolute_error(y_test, y_pred),
        mean_squared_error(y_test, y_pred) ** 0.5,
        r2_score(y_test, y_pred),
    )
)


## 1. Residuals vs fitted (LINE: L + E)

**Healthy look:** cloud centered at 0 with roughly constant vertical spread.

**Red flags**

- Smooth curve / U-shape → linearity miss (wrong functional form)
- Fan / megaphone → heteroscedasticity (unequal variance)

Say out loud: good holdout $R^{2}$ does **not** excuse a structured residual plot.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(y_pred, resid, alpha=0.35, s=12, color="#1971c2")
ax.axhline(0, color="k", lw=1)
ax.set_xlabel(r"Fitted $\hat{y}$")
ax.set_ylabel(r"Residual $y - \hat{y}$")
ax.set_title("Residuals vs fitted")
plt.tight_layout()
plt.show()


## 2. Residuals vs a key feature

If residuals bend against `MedInc` (median income), income's effect is probably not linear in this OLS. Remedies: transforms, polynomial / spline terms, or a nonlinear model.


In [ ]:
feat = "MedInc" if "MedInc" in X_test.columns else (
    "median_income" if "median_income" in X_test.columns else num_cols[0]
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X_test[feat], resid, alpha=0.35, s=12, color="#2f9e44")
ax.axhline(0, color="k", lw=1)
ax.set_xlabel(feat)
ax.set_ylabel("Residual")
ax.set_title(f"Residuals vs {feat}")
plt.tight_layout()
plt.show()


## 3. Normality - histogram + Q-Q

Normality matters most for classical CIs / tests. Large $n$ helps coefficient means via CLT; still plot diagnostics. Heavy tails or skew → be careful with p-values; consider transforms or prediction-only use of OLS.


In [ ]:
import scipy.stats as stats

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(resid, bins=40, color="#1971c2", edgecolor="white")
axes[0].set_title("Residual histogram")
axes[0].set_xlabel("Residual")
axes[0].set_ylabel("Count")

stats.probplot(resid, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot")
plt.tight_layout()
plt.show()


## 4. Independence (quick check)

California housing rows are geographic; perfect IID is optimistic. For this lab we only do a light residual-vs-row-order glance. In production panels / time series: time-aware splits, cluster SE, lagged features.

**CML 2 echo:** leakage and bad splits create fake independence.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(np.arange(len(resid)), resid, lw=0.6, alpha=0.75, color="#e8590c")
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("Test-row order (not time)")
ax.set_ylabel("Residual")
ax.set_title("Residuals vs row order (sanity glance only)")
plt.tight_layout()
plt.show()


## 5. Multicollinearity - correlation heatmap + VIF sketch

Near-collinear features → wild coefficients, huge SE, sign flips when a cousin feature is added.

$$
\mathrm{VIF}_j = \frac{1}{1 - R^{2}_j}
$$

Rule of thumb: VIF above about 5 to 10 deserves a closer look. We compute VIF with a small OLS helper (no statsmodels required).


In [ ]:
corr = X_train.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Feature correlation (train)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

corr.round(2)


In [ ]:
def vif_table(frame: pd.DataFrame) -> pd.DataFrame:
    """VIF via per-column R^2 from LinearRegression on the other columns."""
    cols = list(frame.columns)
    Xmat = frame.to_numpy(dtype=float)
    rows = []
    for j, name in enumerate(cols):
        y_j = Xmat[:, j]
        X_others = np.delete(Xmat, j, axis=1)
        lr = LinearRegression()
        lr.fit(X_others, y_j)
        r2 = lr.score(X_others, y_j)
        r2 = min(r2, 0.999999)  # guard perfect collinearity
        vif = 1.0 / (1.0 - r2)
        rows.append({"feature": name, "R2_others": r2, "VIF": vif})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False).reset_index(drop=True)


# VIF on the *scaled imputed* train matrix (same space the OLS coefs live in)
X_vif = pd.DataFrame(
    model.named_steps["prep"].transform(X_train),
    columns=num_cols,
)
vif_df = vif_table(X_vif)
vif_df


## 6. Optional remedy demo: log-target against a fan

House prices often show heteroscedasticity on the dollar scale. A first-line habit: fit OLS on $\log(y)$, then re-check residuals vs fitted on the log scale.

Interpret log-scale coefficients as approximate multiplicative effects. If stakeholders need dollars, also report metrics after `expm1`.


In [ ]:
# Guard: target must be positive for log1p
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

model_log = Pipeline(
    [
        (
            "prep",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
        ),
        ("lr", LinearRegression()),
    ]
)
model_log.fit(X_train, y_train_log)
y_pred_log = model_log.predict(X_test)
resid_log = y_test_log.to_numpy() - y_pred_log

# dollar-scale predictions for stakeholder metrics
y_pred_dollar = np.expm1(y_pred_log)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(y_pred, resid, alpha=0.3, s=10, color="#1971c2")
axes[0].axhline(0, color="k", lw=1)
axes[0].set_title("Original scale")
axes[0].set_xlabel(r"$\hat{y}$")
axes[0].set_ylabel("Residual")

axes[1].scatter(y_pred_log, resid_log, alpha=0.3, s=10, color="#9c36b5")
axes[1].axhline(0, color="k", lw=1)
axes[1].set_title("log1p target")
axes[1].set_xlabel(r"$\widehat{\log(1+y)}$")
axes[1].set_ylabel("Residual (log scale)")
plt.tight_layout()
plt.show()

print(
    "dollar metrics from expm1(log model): MAE={:.3f}  RMSE={:.3f}  R2={:.3f}".format(
        mean_absolute_error(y_test, y_pred_dollar),
        mean_squared_error(y_test, y_pred_dollar) ** 0.5,
        r2_score(y_test, y_pred_dollar),
    )
)


## 7. Discuss / exit ticket

Which assumption looks most strained on California housing?

Prompts:

1. Is the residual-vs-fitted cloud curved, fanned, or mostly healthy?
2. Does `MedInc` show a bend that a linear term cannot capture?
3. Which features have the highest VIF? Would you drop, combine, or regularize?
4. Did log-$y$ calm the fan? What tradeoff does that create for interpretation?

**First experiments to try next:** log target, inspect high-VIF pairs, polynomial / spline term for income, then (later sessions) Ridge / Lasso when coefficient stability matters.
